# 02 -- Raw-Data Validation

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Walks through severity-graded raw-data validation (`info`/`warning`/`error`/`fatal`) on the synthetic fixture bundle. The same functions run here are what `atlas-quant filing-momentum validate-data` runs against real, acquired data.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
bundle = _fixtures.build_synthetic_bundle()
calendar = _fixtures.build_trading_calendar(bundle)
periods = _fixtures.build_periods()
print("universe:", [i.symbol for i in bundle.universe])
print("normalization issues:", len(bundle.all_issues))


In [ ]:
from atlas_quant.strategies.filing_momentum_ml.production.validation import (
    DataValidationSummary, validate_calendar, validate_filings, validate_prices,
    validate_sectors, validate_universe,
)

issues = list(bundle.all_issues)
for filings in bundle.filings_by_instrument.values():
    issues.extend(validate_filings(filings))
for instrument_id in bundle.universe:
    issues.extend(validate_prices(bundle.prices_by_instrument.get(instrument_id, ())))
issues.extend(validate_sectors(list(bundle.sector_by_instrument.values())))
issues.extend(validate_universe([]))  # demonstrates the FATAL empty-universe case explicitly
summary = DataValidationSummary(issues=tuple(issues))
print("counts by severity:", summary.counts_by_severity())
for issue in summary.issues:
    print(f"  [{issue.severity.value:7s}] {issue.category}/{issue.subject}: {issue.message}")


## Findings

- The synthetic fixture bundle itself is clean (only the survivorship-bias `info` note); the empty-universe call above is included solely to demonstrate the `FATAL` path without editing `_fixtures.py`.

## Limitations

- Real production data will surface categories of issues this small synthetic bundle cannot (duplicate filings across amendments, price gaps from halts/delistings, conflicting sector classifications). This notebook demonstrates the mechanism, not full real-world coverage.